# TASTERA — Baseline Recommenders

This notebook establishes reproducible baseline methods using MovieLens Latest Small.

**Methods**
- Global-mean and user-mean rating predictors, evaluated with MAE and RMSE.
- Popularity-based recommendations.
- Genre-overlap content recommendations.

The rating-prediction metrics do not evaluate the ranking methods. Ranking metrics such as Recall@K and NDCG@K should be added in a later experiment with an explicitly defined relevance rule.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

def find_project_root(start: Path) -> Path:
    """Find the repository root without relying on a machine-specific path."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "tastera" / "recommenders" / "baselines.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the TASTERA repository root. "
        "Run this notebook from inside the cloned repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "movielens" / "ml-latest-small"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

if not (DATA_DIR / "ratings.csv").exists():
    raise FileNotFoundError(f"Ratings file not found: {DATA_DIR / 'ratings.csv'}")

if not (DATA_DIR / "movies.csv").exists():
    raise FileNotFoundError(f"Movies file not found: {DATA_DIR / 'movies.csv'}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tastera.recommenders.baselines import (
    ContentBasedRecommender,
    GlobalMeanRecommender,
    PopularityRecommender,
    UserMeanRecommender,
)

movies = pd.read_csv(DATA_DIR / "movies.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")

print("Project root:", PROJECT_ROOT)
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)

## 1. Validate and split the data

A chronological holdout is used: the 80th-percentile timestamp is the cutoff. Training contains timestamps strictly below the cutoff; test contains timestamps at or after it. This keeps evaluation ratings out of model fitting. Because the split is timestamp-based, some users or movies may be absent from training; the user-mean model falls back to the global mean for unseen users.


In [ ]:
required_rating_cols = {"userId", "movieId", "rating", "timestamp"}
required_movie_cols = {"movieId", "title", "genres"}
assert required_rating_cols.issubset(ratings.columns), required_rating_cols - set(ratings.columns)
assert required_movie_cols.issubset(movies.columns), required_movie_cols - set(movies.columns)
assert ratings["rating"].between(0.5, 5.0).all()
assert (ratings["rating"].mul(2) % 1 == 0).all(), "Expected half-star rating increments"

cutoff = ratings["timestamp"].quantile(0.80)
train = ratings.loc[ratings["timestamp"] < cutoff].copy()
test = ratings.loc[ratings["timestamp"] >= cutoff].copy()

if train.empty or test.empty:
    raise ValueError("Temporal split produced an empty train or test set.")

print(f"Cutoff timestamp: {cutoff}")
print(f"Train rows: {len(train):,} ({len(train)/len(ratings):.1%})")
print(f"Test rows:  {len(test):,} ({len(test)/len(ratings):.1%})")
print(f"Train time range: {train.timestamp.min()} to {train.timestamp.max()}")
print(f"Test time range:  {test.timestamp.min()} to {test.timestamp.max()}")


## 2. Rating-prediction baselines

MAE measures average absolute error; RMSE penalizes larger errors more strongly. Lower values are better. Both models use only the training split.

In [ ]:
rating_models = {
    "global_mean": GlobalMeanRecommender().fit(train),
    "user_mean": UserMeanRecommender().fit(train),
}

metric_rows = []
predictions = {}
for model_name, model in rating_models.items():
    y_pred = model.predict(test["userId"], test["movieId"])
    predictions[model_name] = y_pred
    metric_rows.append({
        "model": model_name,
        "test_rows": len(test),
        "MAE": mean_absolute_error(test["rating"], y_pred),
        "RMSE": np.sqrt(mean_squared_error(test["rating"], y_pred)),
    })

rating_metrics = pd.DataFrame(metric_rows).sort_values("RMSE").reset_index(drop=True)
display(rating_metrics)
rating_metrics.to_csv(OUTPUT_DIR / "baseline_rating_metrics.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'baseline_rating_metrics.csv'}")


## 3. Popularity recommendations

The ranking is fitted on training ratings only. The minimum count is configurable; 20 is a reasonable initial threshold for this dataset, not a universal optimum.

In [ ]:
popularity_model = PopularityRecommender().fit(train, movies, min_ratings=20)
popular = popularity_model.recommend(n=20)
display(popular[["movieId", "title", "genres", "rating_count", "mean_rating"]])
popular.to_csv(OUTPUT_DIR / "baseline_popularity_top20.csv", index=False)


## 4. Genre-content recommendations

The example user is selected from the training data so the demonstration is reproducible. The model uses training ratings to identify liked movies and excludes movies the user has already rated in training.

In [ ]:
example_user_id = int(train["userId"].value_counts().index[0])
content_model = ContentBasedRecommender().fit(movies)
content_recs = content_model.recommend_for_user(
    example_user_id, train, n=20, like_threshold=4.0
)

print(f"Example user ID: {example_user_id}")
print("Highly rated training movies for this user:")
liked_ids = train.loc[
    (train["userId"] == example_user_id) & (train["rating"] >= 4.0), "movieId"
]
display(movies[movies["movieId"].isin(liked_ids)][["title", "genres"]].head(15))
print("Recommended unseen movies:")
display(content_recs[["movieId", "title", "genres", "content_score", "matched_genres"]])
content_recs.to_csv(OUTPUT_DIR / "baseline_content_top20.csv", index=False)


## 5. Interpretation and next steps

- Compare the rating predictors using MAE/RMSE only.
- Inspect the popularity list for head-item bias.
- Inspect the content list to see whether genre overlap is meaningful for the example user.
- Do not claim the content or popularity method is quantitatively superior based on these tables.
- Next experiment: define positive test interactions (for example, ratings >= 4.0), then evaluate ranking metrics such as Precision@K, Recall@K, and NDCG@K with clear eligibility and cold-start handling.


In [ ]:
print("Saved output files:")
for path in sorted(OUTPUT_DIR.glob("baseline_*")):
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")
